<a href="https://colab.research.google.com/github/noorrami/ptbxl-ecg-classification/blob/main/notebooks/03_full_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase 0: Dependencies

In [ ]:
%pip install -q wfdb numpy pandas matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.6 MB/s eta 0:00:00


Phase 1: Environment Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import urlretrieve
from urllib.error import URLError, HTTPError
from concurrent.futures import ThreadPoolExecutor, as_completed

import hashlib
import json
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import wfdb

from tqdm.auto import tqdm
from google.colab import drive

drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content/drive/MyDrive/ML_Master_Project")

PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results" / "full_dataset"
CHUNKS = PROCESSED / "ecg_chunks"

# Fast temporary storage within the current Colab session
CACHE = Path("/content/ptbxl_download_cache")

for path in [PROCESSED, RESULTS, CHUNKS, CACHE]:
    path.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://physionet.org/files/ptb-xl/1.0.3"

FS = 100
N_SAMPLES = 1000
N_LEADS = 12

CHUNK_SIZE = 64
MAX_RETRIES = 3
RETRY_DELAY = 2

EXPECTED_LEADS = [
    "I", "II", "III",
    "AVR", "AVL", "AVF",
    "V1", "V2", "V3",
    "V4", "V5", "V6"
]

print("Project root:", ROOT)
print("Output directory:", CHUNKS)
print("Temporary cache:", CACHE)
print("WFDB version:", wfdb.__version__)

Mounted at /content/drive
Project root: /content/drive/MyDrive/ML_Master_Project
Output directory: /content/drive/MyDrive/ML_Master_Project/data/processed/ecg_chunks
Temporary cache: /content/ptbxl_download_cache
WFDB version: 4.3.1


Phase 2: Validated Manifest Loading

In [ ]:
def load_manifest(name):
    path = PROCESSED / f"{name}_manifest.csv"

    if not path.exists():
        raise FileNotFoundError(
            f"Missing manifest: {path}"
        )

    df = pd.read_csv(path, index_col="ecg_id")

    required = [
        "patient_id",
        "strat_fold",
        "target",
        "filename_lr"
    ]

    if not set(required).issubset(df.columns):
        raise ValueError(f"Missing columns in {name}.")

    if not df.index.is_unique:
        raise ValueError(f"Duplicate ECG IDs in {name}.")

    if df[required].isna().any().any():
        raise ValueError(f"Missing required values in {name}.")

    if not df["target"].isin([0, 1]).all():
        raise ValueError(f"Invalid targets in {name}.")

    return df.sort_index()


train_df = load_manifest("train")
val_df = load_manifest("validation")
test_df = load_manifest("test")

assert train_df["strat_fold"].between(1, 8).all()
assert val_df["strat_fold"].eq(9).all()
assert test_df["strat_fold"].eq(10).all()

assert len(train_df) == 9286
assert len(val_df) == 1147
assert len(test_df) == 1168

for a, b in [
    (train_df, val_df),
    (train_df, test_df),
    (val_df, test_df)
]:
    assert set(a.index).isdisjoint(b.index)
    assert set(a["patient_id"]).isdisjoint(
        set(b["patient_id"])
    )

print("PASS: Official split sizes.")
print("PASS: Patient-disjoint partitions.")
print("PASS: Record-disjoint partitions.")
print("PASS: Test manifest verified without waveform access.")

PASS: Official split sizes.
PASS: Patient-disjoint partitions.
PASS: Record-disjoint partitions.
PASS: Test manifest verified without waveform access.


**Phase 3: Immutable Processing Configuration**



The acquisition pipeline uses a fixed configuration to ensure that every ECG recording is processed under the same technical assumptions.

Record identifiers, patient identifiers, labels, and waveform ordering are preserved throughout processing. Each completed chunk is associated with a configuration fingerprint to prevent accidental reuse of outputs generated under incompatible settings.

Waveform acquisition and quality assessment are intentionally separated from normalization. No training-derived or evaluation-derived normalization parameters are estimated during this phase.


In [ ]:
CONFIG = {
    "dataset": "PTB-XL",
    "version": "1.0.3",
    "sampling_frequency_hz": FS,
    "samples_per_record": N_SAMPLES,
    "number_of_leads": N_LEADS,
    "chunk_size": CHUNK_SIZE,
    "dtype": "float32",
    "storage": "npz_compressed",
    "normalization": "none",
    "seed": SEED
}

CONFIG_ID = hashlib.sha256(
    json.dumps(
        CONFIG,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()[:16]

CONFIG_PATH = RESULTS / "acquisition_config.json"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, encoding="utf-8") as f:
        previous_config = json.load(f)

    if previous_config != CONFIG:
        raise RuntimeError(
            "Existing outputs use a different configuration. "
            "Do not mix runs. Use a new output directory."
        )
else:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(CONFIG, f, indent=4)

print("Configuration ID:", CONFIG_ID)

Configuration ID: c37b7aee98472c39


Phase 4: Safe Download and Retry Logic

In [ ]:
def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    if destination.exists() and destination.stat().st_size > 0:
        return destination

    for attempt in range(1, MAX_RETRIES + 1):
        temp_path = destination.with_suffix(
            destination.suffix + ".part"
        )

        try:
            if temp_path.exists():
                temp_path.unlink()

            urlretrieve(url, temp_path)

            if temp_path.stat().st_size == 0:
                raise IOError("Downloaded file is empty.")

            os.replace(temp_path, destination)

            return destination

        except (HTTPError, URLError, OSError, IOError):
            if temp_path.exists():
                temp_path.unlink()

            if attempt == MAX_RETRIES:
                raise

            time.sleep(RETRY_DELAY * attempt)


**Phase 5: WFDB Signal Validation**

Every ECG recording must satisfy a consistent waveform contract before inclusion in the processed dataset.

The pipeline verifies the sampling frequency, number of time samples, number and ordering of ECG leads, and numerical finiteness of the signal.

These checks establish structural and numerical integrity. They do not establish clinical signal quality, diagnostic correctness, or freedom from physiological and acquisition artifacts.

Record-level anomalies are documented rather than silently corrected or excluded.


In [ ]:
def load_and_validate_ecg(relative_path):
    relative_path = str(relative_path)

    if not relative_path.startswith("records100/"):
        raise ValueError(
            f"Unexpected path: {relative_path}"
        )

    local_base = CACHE / relative_path

    for extension in [".hea", ".dat"]:
        relative_file = relative_path + extension

        download_file(
            f"{BASE_URL}/{relative_file}",
            CACHE / relative_file
        )

    try:
        record = wfdb.rdrecord(str(local_base))

        signal = np.asarray(
            record.p_signal,
            dtype=np.float32
        )

        if record.fs != FS:
            raise ValueError(
                f"Unexpected sampling rate: {record.fs}"
            )

        if signal.shape != (N_SAMPLES, N_LEADS):
            raise ValueError(
                f"Unexpected shape: {signal.shape}"
            )

        leads = [
            lead.upper()
            for lead in record.sig_name
        ]

        if leads != EXPECTED_LEADS:
            raise ValueError(
                f"Unexpected lead order: {leads}"
            )

        if not np.isfinite(signal).all():
            raise ValueError(
                "Non-finite waveform values detected."
            )

        return signal

    finally:
        # Raw files are temporary; processed chunks persist on Drive.
        for extension in [".hea", ".dat"]:
            file_path = Path(str(local_base) + extension)

            if file_path.exists():
                file_path.unlink()

Phase 6: Signal Quality Metrics

In [ ]:
def calculate_quality_metrics(signal, ecg_id):
    channel_std = np.std(signal, axis=0)

    return {
        "ecg_id": int(ecg_id),
        "min_amplitude": float(np.min(signal)),
        "max_amplitude": float(np.max(signal)),
        "max_abs_amplitude": float(
            np.max(np.abs(signal))
        ),
        "constant_channels": int(
            np.sum(np.ptp(signal, axis=0) == 0)
        ),
        "min_channel_std": float(
            np.min(channel_std)
        ),
        "max_channel_std": float(
            np.max(channel_std)
        ),
        "finite": bool(np.isfinite(signal).all())
    }


**Phase 7: Chunk Processing and Atomic Storage**

The dataset is processed in deterministic chunks to support resumable execution and reduce peak memory consumption.

Each chunk contains ECG waveforms, class labels, patient identifiers, and ECG record identifiers in a consistent order.

A chunk is committed only after all expected recordings have been retrieved and validated. Temporary output files are replaced atomically after successful writing, reducing the risk of treating partially written files as completed results.

Processing failures are recorded explicitly. Incomplete chunks are not accepted as valid dataset artifacts.


In [ ]:
def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while block := f.read(block_size):
            digest.update(block)

    return digest.hexdigest()


def chunk_paths(split_name, chunk_index):
    folder = CHUNKS / split_name
    folder.mkdir(parents=True, exist_ok=True)

    prefix = f"chunk_{chunk_index:04d}"

    return {
        "data": folder / f"{prefix}.npz",
        "checksum": folder / f"{prefix}.sha256",
        "qc": folder / f"{prefix}_qc.csv",
        "errors": folder / f"{prefix}_errors.csv"
    }


def validate_saved_chunk(path, expected_ids):
    with np.load(path, allow_pickle=False) as data:
        required = {
            "X", "y", "ecg_ids",
            "patient_ids", "config_id"
        }

        if not required.issubset(data.files):
            return False

        X = data["X"]
        y = data["y"]
        ids = data["ecg_ids"]
        patient_ids = data["patient_ids"]

        if str(data["config_id"].item()) != CONFIG_ID:
            return False

        if X.shape != (len(expected_ids), 1000, 12):
            return False

        if X.dtype != np.float32:
            return False

        if not np.array_equal(ids, expected_ids):
            return False

        if len(y) != len(ids) or len(patient_ids) != len(ids):
            return False

        if not np.isfinite(X).all():
            return False

        return True


def process_chunk(split_name, chunk_index, chunk_df):
    paths = chunk_paths(split_name, chunk_index)
    expected_ids = chunk_df.index.to_numpy(
        dtype=np.int64
    )

    if paths["data"].exists():
        checksum_valid = (
            paths["checksum"].exists()
            and paths["checksum"].read_text().strip()
                == sha256_file(paths["data"])
        )

        if checksum_valid and validate_saved_chunk(
            paths["data"], expected_ids
        ):
            return "skipped"

        raise RuntimeError(
            f"Existing chunk failed verification: "
            f"{paths['data']}. Investigate before retrying."
        )

    signals = []
    labels = []
    record_ids = []
    patient_ids = []
    qc_rows = []
    error_rows = []

    for ecg_id, row in chunk_df.iterrows():
        try:
            signal = load_and_validate_ecg(
                row["filename_lr"]
            )

            signals.append(signal)
            labels.append(int(row["target"]))
            record_ids.append(int(ecg_id))
            patient_ids.append(int(row["patient_id"]))

            qc_rows.append(
                calculate_quality_metrics(signal, ecg_id)
            )

        except Exception as exc:
            error_rows.append({
                "ecg_id": int(ecg_id),
                "filename_lr": str(row["filename_lr"]),
                "error_type": type(exc).__name__,
                "error": str(exc)
            })

    if error_rows:
        pd.DataFrame(error_rows).to_csv(
            paths["errors"],
            index=False
        )

        return "failed"

    X = np.stack(signals).astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)
    ids = np.asarray(record_ids, dtype=np.int64)
    patients = np.asarray(patient_ids, dtype=np.int64)

    temp_path = paths["data"].with_suffix(".tmp.npz")

    try:
        np.savez_compressed(
            temp_path,
            X=X,
            y=y,
            ecg_ids=ids,
            patient_ids=patients,
            config_id=np.array(CONFIG_ID)
        )

        assert validate_saved_chunk(
            temp_path, expected_ids
        )

        checksum = sha256_file(temp_path)

        os.replace(temp_path, paths["data"])

        paths["checksum"].write_text(
            checksum,
            encoding="utf-8"
        )

        pd.DataFrame(qc_rows).to_csv(
            paths["qc"],
            index=False
        )

        if paths["errors"].exists():
            paths["errors"].unlink()

    finally:
        if temp_path.exists():
            temp_path.unlink()

    return "completed"

Phase 8: Manifest-Driven Chunk Execution

In [ ]:
def process_split(split_name, dataframe, max_chunks=None):
    n_chunks = (
        len(dataframe) + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    if max_chunks is not None:
        n_chunks = min(n_chunks, max_chunks)

    results = {
        "completed": 0,
        "skipped": 0,
        "failed": 0
    }

    for chunk_index in tqdm(
        range(n_chunks),
        desc=f"Processing {split_name}"
    ):
        start = chunk_index * CHUNK_SIZE
        stop = start + CHUNK_SIZE

        chunk_df = dataframe.iloc[start:stop]

        status = process_chunk(
            split_name,
            chunk_index,
            chunk_df
        )

        results[status] += 1

        if status == "failed":
            print(
                f"Chunk {chunk_index:04d} failed. "
                "Check its error report."
            )

    print(f"\n{split_name} summary:", results)

    return results

Phase 9: Acquisition Smoke Test

In [ ]:
smoke_result = process_split(
    "train",
    train_df,
    max_chunks=1
)

assert smoke_result["failed"] == 0

print("PASS: First acquisition chunk completed.")

Processing train:   0%|          | 0/1 [00:00<?, ?it/s]


train summary: {'completed': 1, 'skipped': 0, 'failed': 0}
PASS: First acquisition chunk completed.


Phase 10: Saved Chunk Inspection

In [ ]:
sample_path = chunk_paths(
    "train", 0
)["data"]

with np.load(
    sample_path,
    allow_pickle=False
) as data:

    print("Signals:", data["X"].shape)
    print("Labels:", data["y"].shape)
    print("ECG IDs:", data["ecg_ids"].shape)
    print("Patient IDs:", data["patient_ids"].shape)
    print("Signal dtype:", data["X"].dtype)
    print("Class counts:", np.bincount(data["y"]))

assert validate_saved_chunk(
    sample_path,
    train_df.index.to_numpy(dtype=np.int64)[:64]
)

print("PASS: Persisted chunk integrity.")

Signals: (64, 1000, 12)
Labels: (64,)
ECG IDs: (64,)
Patient IDs: (64,)
Signal dtype: float32
Class counts: [64]
PASS: Persisted chunk integrity.


Phase 11: Resume Capability Test

In [ ]:
resume_result = process_split(
    "train",
    train_df,
    max_chunks=1
)

assert resume_result["skipped"] == 1

print("PASS: Completed chunk safely reused.")

Processing train:   0%|          | 0/1 [00:00<?, ?it/s]


train summary: {'completed': 0, 'skipped': 1, 'failed': 0}
PASS: Completed chunk safely reused.



**Phase 12: Metadata Alignment Verification**

Correct waveform dimensions alone do not guarantee dataset integrity. Every stored waveform must remain aligned with its original ECG identifier, patient identifier, and diagnostic target.

The verification procedure compares persisted metadata against the authoritative split manifest. This protects against silent label misalignment, incorrect sample ordering, and accidental reuse of incompatible chunks.

All records must be accounted for before the processed dataset can be considered complete.


In [ ]:
def verify_chunk_alignment(split_name, chunk_index, dataframe):
    paths = chunk_paths(split_name, chunk_index)

    start = chunk_index * CHUNK_SIZE
    subset = dataframe.iloc[start:start + CHUNK_SIZE]

    with np.load(paths["data"], allow_pickle=False) as data:
        np.testing.assert_array_equal(
            data["ecg_ids"],
            subset.index.to_numpy(dtype=np.int64)
        )

        np.testing.assert_array_equal(
            data["patient_ids"],
            subset["patient_id"].to_numpy(dtype=np.int64)
        )

        np.testing.assert_array_equal(
            data["y"],
            subset["target"].to_numpy(dtype=np.int64)
        )

    assert paths["checksum"].read_text().strip() == (
        sha256_file(paths["data"])
    )

    assert paths["qc"].exists()

    qc = pd.read_csv(paths["qc"])

    assert len(qc) == len(subset)
    np.testing.assert_array_equal(
        qc["ecg_id"].to_numpy(dtype=np.int64),
        subset.index.to_numpy(dtype=np.int64)
    )

    return True


assert verify_chunk_alignment(
    "train", 0, train_df
)

print("PASS: ECG, patient, and label alignment.")

PASS: ECG, patient, and label alignment.


In [14]:
def verify_existing_chunks(split_name, dataframe):
    n_chunks = (
        len(dataframe) + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    verified = 0

    for chunk_index in range(n_chunks):
        paths = chunk_paths(split_name, chunk_index)

        if not paths["data"].exists():
            continue

        if not paths["checksum"].exists():
            raise RuntimeError(
                f"Missing checksum: {paths['data']}"
            )

        if not verify_chunk_alignment(
            split_name,
            chunk_index,
            dataframe
        ):
            raise RuntimeError(
                f"Metadata mismatch: {paths['data']}"
            )

        verified += 1

    return verified


verified_train = verify_existing_chunks(
    "train", train_df
)

verified_val = verify_existing_chunks(
    "validation", val_df
)

print("Verified training chunks:", verified_train)
print("Verified validation chunks:", verified_val)
print("PASS: Existing chunks match source manifests.")

Verified training chunks: 128
Verified validation chunks: 0
PASS: Existing chunks match source manifests.


Phase 13: Full Train and Validation Acquisition

In [ ]:
train_result = process_split(
    "train",
    train_df
)

val_result = process_split(
    "validation",
    val_df
)

assert train_result["failed"] == 0
assert val_result["failed"] == 0

print("PASS: All acquisition chunks completed.")

Processing train:   0%|          | 0/146 [00:00<?, ?it/s]

Phase 13A: Recovery — Inspect Saved Progress

In [ ]:
from pathlib import Path

train_folder = CHUNKS / "train"
val_folder = CHUNKS / "validation"

train_chunks = sorted(train_folder.glob("chunk_*.npz"))
val_chunks = sorted(val_folder.glob("chunk_*.npz"))

print("Saved training chunks:", len(train_chunks))
print("Saved validation chunks:", len(val_chunks))

print("Estimated saved training records:",
      min(len(train_chunks) * CHUNK_SIZE, len(train_df)))

print("Remaining training chunks:",
      146 - len(train_chunks))

if train_chunks:
    print("Last saved training chunk:",
          train_chunks[-1].name)

Saved training chunks: 128
Saved validation chunks: 0
Estimated saved training records: 8192
Remaining training chunks: 18
Last saved training chunk: chunk_0127.npz


Phase 13C: Verify Recovered Training Chunks

In [15]:
verified_train = verify_existing_chunks(
    "train",
    train_df
)

print("Verified training chunks:", verified_train)

assert verified_train == 128
print("PASS: All 128 saved chunks verified.")

Verified training chunks: 128
PASS: All 128 saved chunks verified.


Phase 13D: Resume Missing Training Chunks

In [16]:
from tqdm.auto import tqdm

def resume_missing_chunks(split_name, dataframe):
    total_chunks = (
        len(dataframe) + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    results = {
        "completed": 0,
        "skipped": 0,
        "failed": 0
    }

    for chunk_index in tqdm(
        range(total_chunks),
        desc=f"Resuming {split_name}"
    ):
        start = chunk_index * CHUNK_SIZE
        stop = start + CHUNK_SIZE

        subset = dataframe.iloc[start:stop]

        status = process_chunk(
            split_name,
            chunk_index,
            subset
        )

        results[status] += 1

        if status == "failed":
            print(f"Failed chunk: {chunk_index:04d}")

    print(f"\n{split_name} results:", results)
    return results


train_result = resume_missing_chunks(
    "train",
    train_df
)

assert train_result["failed"] == 0
print("PASS: Training acquisition completed.")

Resuming train:   0%|          | 0/146 [00:00<?, ?it/s]


train results: {'completed': 18, 'skipped': 128, 'failed': 0}
PASS: Training acquisition completed.


Phase 13E: Validation Dataset Acquisition

In [17]:
val_result = resume_missing_chunks(
    "validation",
    val_df
)

assert val_result["failed"] == 0

print("PASS: Validation acquisition completed.")

Resuming validation:   0%|          | 0/18 [00:00<?, ?it/s]


validation results: {'completed': 18, 'skipped': 0, 'failed': 0}
PASS: Validation acquisition completed.


Phase 14: Full Dataset Completeness Audit

In [18]:
def audit_processed_split(split_name, dataframe):
    n_chunks = (
        len(dataframe) + CHUNK_SIZE - 1
    ) // CHUNK_SIZE

    for chunk_index in tqdm(
        range(n_chunks),
        desc=f"Auditing {split_name}"
    ):
        paths = chunk_paths(split_name, chunk_index)

        assert paths["data"].exists()
        assert paths["checksum"].exists()

        start = chunk_index * CHUNK_SIZE
        expected_ids = dataframe.index.to_numpy(
            dtype=np.int64
        )[start:start + CHUNK_SIZE]

        assert validate_saved_chunk(
            paths["data"],
            expected_ids
        )

        assert verify_chunk_alignment(
            split_name,
            chunk_index,
            dataframe
        )

    return {
        "split": split_name,
        "records": len(dataframe),
        "chunks": n_chunks,
        "verified": True
    }


train_audit = audit_processed_split(
    "train", train_df
)

val_audit = audit_processed_split(
    "validation", val_df
)

audit_report = {
    "dataset": "PTB-XL v1.0.3",
    "config_id": CONFIG_ID,
    "train": train_audit,
    "validation": val_audit,
    "test_waveforms_accessed": False,
    "status": "complete"
}

with open(
    RESULTS / "full_acquisition_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(audit_report, f, indent=4)

print(json.dumps(audit_report, indent=4))

Auditing train:   0%|          | 0/146 [00:00<?, ?it/s]

Auditing validation:   0%|          | 0/18 [00:00<?, ?it/s]

{
    "dataset": "PTB-XL v1.0.3",
    "config_id": "c37b7aee98472c39",
    "train": {
        "split": "train",
        "records": 9286,
        "chunks": 146,
        "verified": true
    },
    "validation": {
        "split": "validation",
        "records": 1147,
        "chunks": 18,
        "verified": true
    },
    "test_waveforms_accessed": false,
    "status": "complete"
}
